# Homework: Building a Wikipedia Agent

Same pattern as the course lessons: plain Python functions with docstrings,
registered as tools on a ToyAIKit agent. Each step below is its own cell,
so you can run one at a time and see the result before moving on.

**Before running:**
- `pip install requests toyaikit openai`
- set your `OPENAI_API_KEY` as an environment variable

In [6]:
import os
import requests

assert "OPENAI_API_KEY" in os.environ, "Set your OPENAI_API_KEY before continuing"

HEADERS = {"User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"}
print("Setup OK")

Setup OK


## Question 1 — Define the search tool

`search_wikipedia` calls the Wikipedia Search API and returns the raw JSON response.

In [7]:
def search_wikipedia(query: str) -> dict:
    """
    Search Wikipedia for pages matching a query.

    Args:
        query (str): The search term.

    Returns:
        dict: The raw Wikipedia search API response.
    """
    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "format": "json",
        "list": "search",
        "srsearch": query,
    }
    response = requests.get(url, params=params, headers=HEADERS)
    response.raise_for_status()
    return response.json()


print("search_wikipedia defined")

search_wikipedia defined


In [8]:
# test it
results = search_wikipedia("capybara")
results

{'batchcomplete': '',
 'continue': {'sroffset': 10, 'continue': '-||'},
 'query': {'searchinfo': {'totalhits': 926},
  'search': [{'ns': 0,
    'title': 'Capybara',
    'pageid': 6776,
    'size': 44449,
    'wordcount': 4312,
    'snippet': 'The <span class="searchmatch">capybara</span> (/kæpɪˈbɑːrə/ kap-ih-BAR-uh) or greater <span class="searchmatch">capybara</span> (Hydrochoerus hydrochaeris) is the largest living rodent, native to all countries in South',
    'timestamp': '2026-09-25T20:39:36Z'},
   {'ns': 0,
    'title': 'Lesser capybara',
    'pageid': 23188846,
    'size': 5681,
    'wordcount': 582,
    'snippet': 'The lesser <span class="searchmatch">capybara</span> (Hydrochoerus isthmius) is a large semi-aquatic rodent found in South America that has vast similarities with, yet subtle differences',
    'timestamp': '2026-08-14T00:29:52Z'},
   {'ns': 0,
    'title': 'Capybara (disambiguation)',
    'pageid': 69085306,
    'size': 535,
    'wordcount': 101,
    'snippet': 'Look

In [9]:
# Q1 answer: total number of results reported by the API
total_hits = results["query"]["searchinfo"]["totalhits"]
print("Q1 - total results:", total_hits)

Q1 - total results: 926


## Question 2 — Analyzing search results

Look at the titles returned and count how many contain "capybara" (case-insensitive).

In [10]:
titles = [r["title"] for r in results["query"]["search"]]
titles

['Capybara',
 'Lesser capybara',
 'Capybara (disambiguation)',
 'Capybara Games',
 'Capybara Party',
 'Capybara (software)',
 'Hydrochoerus',
 'Caviidae',
 'Flow (2024 film)',
 'Yuzu bath']

In [ ]:
matching = [t for t in titles if "capybara" in t.lower()]
print("Matching titles:", matching)
print(f"\nQ2 - {len(matching)} out of {len(titles)} titles contain \'capybara\'")

## Question 3 — Define the get_page tool

`get_page` fetches the raw wikitext content of a single page by title.

In [11]:
def get_page(title: str) -> str:
    """
    Fetch the raw wikitext content of a Wikipedia page.

    Args:
        title (str): The exact title of the page.

    Returns:
        str: The page's raw content.
    """
    url = "https://en.wikipedia.org/w/index.php"
    params = {"title": title, "action": "raw"}
    response = requests.get(url, params=params, headers=HEADERS)
    response.raise_for_status()
    return response.text


print("get_page defined")

get_page defined


In [12]:
# test it
content = get_page("Capybara")
content[:500]  # preview the first 500 characters

'{{Short description|Largest species of rodents}}\n{{Other uses}}\n{{Good article}}\n{{pp|small=yes}}\n{{Use dmy dates|date=July 2022}}\n{{Speciesbox\n| status = LC\n| status_system = IUCN3.1\n| status_ref = <ref name="iucn">{{cite iucn |author=Reid, F. |year=2016 |title=\'\'Hydrochoerus hydrochaeris\'\' |article-number=e.T10300A22190005 |doi=10.2305/IUCN.UK.2016-2.RLTS.T10300A22190005.en}}</ref>\n| fossil_range = {{Fossilrange|Late Pleistocene|Present}}\n| image = Capybaracropped.jpg\n| image_caption = In [[En'

In [13]:
print("Q3 - characters in the Capybara page:", len(content))

Q3 - characters in the Capybara page: 44313


## Question 4 — Agent setup

**Framework:** ToyAIKit (same one used throughout the course)
**LLM provider:** OpenAI, model `gpt-4o-mini`

Register both tools and build the agent.

In [14]:
from toyaikit.tools import Tools

agent_tools = Tools()
agent_tools.add_tool(search_wikipedia)
agent_tools.add_tool(get_page)

agent_tools.get_tools()

[{'type': 'function',
  'name': 'search_wikipedia',
  'description': 'Search Wikipedia for pages matching a query.\n\nArgs:\n    query (str): The search term.\n\nReturns:\n    dict: The raw Wikipedia search API response.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}},
 {'type': 'function',
  'name': 'get_page',
  'description': "Fetch the raw wikitext content of a Wikipedia page.\n\nArgs:\n    title (str): The exact title of the page.\n\nReturns:\n    str: The page's raw content.",
  'parameters': {'type': 'object',
   'properties': {'title': {'type': 'string',
     'description': 'title parameter'}},
   'required': ['title'],
   'additionalProperties': False}}]

In [15]:
instructions = """
You\'re a research assistant that answers questions using Wikipedia.

Use the search_wikipedia tool to find relevant pages, and the get_page
tool to read a page\'s full content before answering.

Use only facts found in the pages you fetch with get_page. If you can\'t
find the answer, say so.
"""

print("Instructions ready")

Instructions ready


In [16]:
from openai import OpenAI
from toyaikit.llm import OpenAIClient
from toyaikit.chat.runners import OpenAIResponsesRunner
from toyaikit.chat.interface import IPythonChatInterface
from toyaikit.chat.runners import DisplayingRunnerCallback

llm_client = OpenAIClient(
    model="gpt-4o-mini",
    client=OpenAI()
)

chat_interface = IPythonChatInterface()
runner_callback = DisplayingRunnerCallback(chat_interface=chat_interface)

agent = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client
)

print("Agent ready")

Agent ready


## Question 5 — Testing the agent: single page

Ask about the Capybara page. Run the cell and watch the callback output below it —
you'll see the `get_page` tool call happen live, then the agent's summary.

In [17]:
result5 = agent.loop(
    "What is this page about? https://en.wikipedia.org/wiki/Capybara",
    callback=runner_callback
)

-> Response received


-> Response received


In [18]:
print("Q5 - agent\'s answer:")
print(result5.last_message)

Q5 - agent's answer:
The Wikipedia page on the **capybara** (scientific name: *Hydrochoerus hydrochaeris*) provides comprehensive information about this large rodent, which is native to South America, excluding Chile. Here are the key points about capybaras:

1. **Physical Description**: Capybaras are recognized as the largest living rodents, with a barrel-shaped body, broad head, and reddish-brown to yellowish-brown fur. They typically weigh between 35 to 66 kg (77 to 146 lbs) and can grow up to 134 cm (53 inches) in length.

2. **Habitat**: They are semiaquatic and prefer living near freshwater bodies such as rivers, lakes, and swamps. Capybaras inhabit savannas and dense forests across various countries in South America.

3. **Social Behavior**: Known for being social creatures, they usually live in groups of 10 to 20 individuals, though larger gatherings can occur, particularly in dry seasons.

4. **Diet**: As herbivores, capybaras primarily eat grasses and aquatic plants but also 

## Question 6 — Testing the agent: search then fetch

Ask about threats to capybara populations. This should make the agent use
`search_wikipedia` first, then `get_page` on the relevant page(s).

In [19]:
result6 = agent.loop(
    "What are the main threats to capybara populations?",
    callback=runner_callback
)

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


In [20]:
print("Q6 - agent\'s answer:")
print(result6.last_message)

Q6 - agent's answer:
The main threats to capybara populations include:

1. **Habitat Loss**: Expansion of agriculture, urban development, and deforestation reduce available habitats for capybaras.

2. **Hunting**: Capybaras are hunted for their meat and skin, which can lead to population declines, especially in regions where hunting is intense.

3. **Human Activity**: Human encroachment often leads to habitat fragmentation, making it difficult for capybaras to find mates and establish stable populations.

4. **Climate Change**: Changes in weather patterns can affect their wetland habitats, impacting their food source availability and breeding success.

5. **Predation**: Increased predation from natural predators, like jaguars, can also affect population numbers, particularly in areas where capybaras are more easily targeted.

These threats are exacerbated by the capybara’s preference for habitats that are often in conflict with human development. As a result, conservation efforts are n

In [ ]:
# count and list every tool call the agent made, in order
tool_calls = [
    m for m in result6.all_messages
    if getattr(m, "type", None) == "function_call"
]

print(f"Q6 - total tool calls made: {len(tool_calls)}\n")
for i, call in enumerate(tool_calls, 1):
    print(f"{i}. {call.name}({call.arguments})")

---

### Recap of answers

Run all the cells above, then read the printed answers off each step:

- **Q1** — printed after the `search_wikipedia("capybara")` test
- **Q2** — printed after listing and filtering the titles
- **Q3** — printed after `get_page("Capybara")`
- **Q4** — ToyAIKit + OpenAI `gpt-4o-mini` (stated above, not computed)
- **Q5** — the agent's printed answer about the Capybara page
- **Q6** — the agent's printed answer + the tool-call count and order

Exact numbers (Q1–Q3) and the agent's exact wording (Q5–Q6) depend on
Wikipedia's current content and the model's own behavior, so don't expect
identical results to someone else's run — the homework says to pick the
closest option if that happens.